In [3]:
# ============================================================
# Project root & path handling
# ------------------------------------------------------------
# Default: working directory
# ============================================================

from pathlib import Path
import os
import json
import numpy as np
import pandas as pd

# Determine project root
PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", ".")).expanduser().resolve()

# Convenience function for building repo-relative paths
def p(rel_path):
    """
    Build an absolute path from a path relative to the project root.
    """
    return PROJECT_ROOT / rel_path

print("PROJECT_ROOT set to:", PROJECT_ROOT)

PROJECT_ROOT set to: /net/bbi/vol1/home/mtejura/IGVF-cvfg-pillar-project


In [10]:
# ============================================================
# Create temporary and output directories 
# ============================================================

DATA_DIR = p("data")
TMP_DIR = p("tmp")
OUT_DIR = p("outputs")

TMP_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

In [11]:
import pandas as pd

sankey_OP = pd.read_csv(OUT_DIR/"integrated_variant_effect_dataset_analysis.csv.gz")

/tmp/7612373.1.fowler-login.q/ipykernel_2183752/849335129.py:3: DtypeWarning: Columns (4,11,12,23,24,30,31,32,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,54,56,58,60,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,90,91,92,97,98,99,100,128,129,130,131,132,133,134,135,136,146,148,151) have mixed types. Specify dtype option on import or set low_memory=False.
  sankey_OP = pd.read_csv(OUT_DIR/"integrated_variant_effect_dataset_analysis.csv.gz")


In [12]:
sankey_OP_2 = sankey_OP[sankey_OP['Gene'] != 'SFPQ']

In [13]:
sankey_OP_2['VariantNotes_OP'] = np.where(
    sankey_OP_2['splice_var_amino'] == 'Yes',
    'splice_variant',
    ''
)

/tmp/7612373.1.fowler-login.q/ipykernel_2183752/3558294941.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sankey_OP_2['VariantNotes_OP'] = np.where(


In [14]:
start_lost = (
    (sankey_OP_2['nucleotide_or_aa'] == 'aa') &
    (sankey_OP_2['simplified_consequence'] == 'start_lost')
)

tag = 'start_lost_variant_not_measured'

existing = sankey_OP_2.loc[start_lost, 'VariantNotes_OP'].fillna('').astype(str)

sankey_OP_2.loc[start_lost, 'VariantNotes_OP'] = np.where(
    existing != "",
    existing + ';' + tag,
    tag
)

In [15]:
OP_nuc = sankey_OP_2[sankey_OP_2['nucleotide_or_aa'] == 'nucleotide']

OP_aa = sankey_OP_2[sankey_OP_2['nucleotide_or_aa'] == 'aa']

In [16]:
#mark any variants that get the opposite evidence between two assays

import numpy as np

group_cols = ['Gene', 'Chrom', 'hg38_start', 'ref_allele', 'alt_allele']

OP_nuc['Chrom'] = OP_nuc['Chrom'].astype(str)
OP_nuc['hg38_start'] = OP_nuc['hg38_start'].astype(str)
OP_nuc['ref_allele'] = OP_nuc['ref_allele'].astype(str)
OP_nuc['alt_allele'] = OP_nuc['alt_allele'].astype(str)
OP_nuc['Gene'] = OP_nuc['Gene'].astype(str)


def has_opposite_signs(x):
    x = x.dropna()
    non_zero = x[x != 0]
    return (non_zero > 0).any() and (non_zero < 0).any()

#2018 clinvar conflicting

conflict_mask_OP_18 = OP_nuc.groupby(group_cols)['OP_points'] \
    .transform(lambda x: has_opposite_signs(x))

conflict_mask_OP_18 = conflict_mask_OP_18.fillna(False)

mask_2_18_OP = conflict_mask_OP_18 & OP_nuc['VariantNotes_OP'].notna() & (OP_nuc['VariantNotes_OP'] != "")

OP_nuc.loc[mask_2_18_OP, 'VariantNotes_OP'] = OP_nuc.loc[mask_2_18_OP, 'VariantNotes_OP'] + ';conflicting_fxn_data'

OP_nuc.loc[conflict_mask_OP_18 & ~mask_2_18_OP, 'VariantNotes_OP'] = 'conflicting_fxn_data'

def get_first_abs_max_idx(x):
    # Treat NaN as 0
    x_filled = x.fillna(0)

    # Compute the max absolute value
    abs_max = x_filled.abs().max()

    # Find the FIRST index where abs value equals abs_max
    return x_filled[x_filled.abs() == abs_max].index[0]

idx_max_18_OP = OP_nuc.groupby(group_cols)['OP_points'].apply(
    lambda x: get_first_abs_max_idx(x)
)

#restrict to rows where Fxn_use_variant is NA/empty
mask_na_18_OP = OP_nuc['VariantNotes_OP'].isna() | (OP_nuc['VariantNotes_OP'] == "")

OP_nuc.loc[idx_max_18_OP[idx_max_18_OP.isin(OP_nuc[mask_na_18_OP].index)], 'VariantNotes_OP'] = 'first_max_fxn_pts'

/tmp/7612373.1.fowler-login.q/ipykernel_2183752/1082815504.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  OP_nuc['Chrom'] = OP_nuc['Chrom'].astype(str)
/tmp/7612373.1.fowler-login.q/ipykernel_2183752/1082815504.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  OP_nuc['hg38_start'] = OP_nuc['hg38_start'].astype(str)
/tmp/7612373.1.fowler-login.q/ipykernel_2183752/1082815504.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_

In [17]:
OP_aa['Ref_seq_transcript_ID_stripped'] = OP_aa['RefSeq Transcript ID'].str.replace(r'\.\d+$', '', regex=True)

OP_aa['aa_pos'] = pd.to_numeric(OP_aa['aa_pos'], errors='coerce')
OP_aa['aa_ref'] = OP_aa['aa_ref'].astype(str)
OP_aa['aa_alt'] = OP_aa['aa_alt'].astype(str)
OP_aa['Gene'] = OP_aa['Gene'].astype(str)
OP_aa['Ref_seq_transcript_ID_stripped'] = OP_aa['Ref_seq_transcript_ID_stripped'].astype(str)

group_cols_aa = ['Gene', 'aa_ref', 'aa_pos', 'aa_alt','Ref_seq_transcript_ID_stripped']

def has_opposite_signs(x):
    x = x.dropna()
    non_zero = x[x != 0]
    return (non_zero > 0).any() and (non_zero < 0).any()


conflict_mask_aa_18_OP = OP_aa.groupby(group_cols_aa)['OP_points'] \
    .transform(lambda x: has_opposite_signs(x))

conflict_mask_aa_18_OP = conflict_mask_aa_18_OP.fillna(False)

mask_aa_18_OP = conflict_mask_aa_18_OP & OP_aa['VariantNotes_OP'].notna() & (OP_aa['VariantNotes_OP'] != "")

OP_aa.loc[mask_aa_18_OP, 'VariantNotes_OP'] = OP_aa.loc[mask_aa_18_OP, 'VariantNotes_OP'] + ';conflicting_fxn_data'

OP_aa.loc[conflict_mask_aa_18_OP & ~mask_aa_18_OP, 'VariantNotes_OP'] = 'conflicting_fxn_data'


idx_max_aa_18_OP = OP_aa.groupby(group_cols_aa)['OP_points'].apply(
    lambda x: x.fillna(0)[x.fillna(0).abs() == x.fillna(0).abs().max()].index
).explode()

# convert to Index
idx_max_aa_18_OP = pd.Index(idx_max_aa_18_OP)

mask_na_aa_18_OP = OP_aa['VariantNotes_OP'].isna() | (OP_aa['VariantNotes_OP'] == "")

OP_aa.loc[idx_max_aa_18_OP.intersection(OP_aa[mask_na_aa_18_OP].index), 'VariantNotes_OP'] = 'max_fxn_pts'

/tmp/7612373.1.fowler-login.q/ipykernel_2183752/2805124865.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  OP_aa['Ref_seq_transcript_ID_stripped'] = OP_aa['RefSeq Transcript ID'].str.replace(r'\.\d+$', '', regex=True)
/tmp/7612373.1.fowler-login.q/ipykernel_2183752/2805124865.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  OP_aa['aa_pos'] = pd.to_numeric(OP_aa['aa_pos'], errors='coerce')
/tmp/7612373.1.fowler-login.q/ipykernel_2183752/2805124865.py:4: SettingWithCopyWarning: 
A value is trying to be

In [18]:
OP_sankey_full = pd.concat([OP_nuc, OP_aa])

In [22]:
#take out conflicting functional data and splice variants that are not measured 

dis = ['conflicting_fxn_data',
       'splice_variant_not_measured',
       'splice_variant_not_measured;conflicting_fxn_data','start_lost_variant_not_measured']

sankey_OP_18 = OP_sankey_full[
    ~OP_sankey_full['VariantNotes_OP'].isin(dis) &
    (OP_sankey_full['splice_var_amino'] != 'Yes')
]

In [23]:
sankey_OP_18 = sankey_OP_18[sankey_OP_18['Flag'] != '*']

In [24]:
controls_OP_18 = sankey_OP_18[sankey_OP_18['clnsig_group_18_25'].isin(['Benign','Benign/Likely benign','Likely benign','Pathogenic',
                                       'Pathogenic/Likely pathogenic','Likely pathogenic'])]

In [25]:
import numpy as np

one_plus_stars = [
    'criteria provided, single submitter',
    'criteria provided, multiple submitters, no conflicts',
    'reviewed by expert panel',
    'criteria provided, conflicting classifications'
]

priority_genes = ['BRCA1', 'PTEN', 'MSH2', 'TP53']

#create new column with clinvar stars for genes where 2018 calibrations are needed, and if not then 2025 
controls_OP_18['clinvar_star_18_25'] = np.where(
    controls_OP_18['Gene'].isin(priority_genes),  
    controls_OP_18['clinvar_star_2018'], 
    controls_OP_18['clinvar_star_2025']
)

/tmp/7612373.1.fowler-login.q/ipykernel_2183752/97044640.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  controls_OP_18['clinvar_star_18_25'] = np.where(


In [26]:
#remove clinvar conflicts and splice variants 

controls_OP_18_x2 = controls_OP_18[(controls_OP_18['clinvar_conflict_flag_18_25'] != 'has clinvar conflict') & 
(controls_OP_18['splice_var_amino'] != 'Yes')]

In [27]:
controls_nuc_OP_18 = controls_OP_18_x2[controls_OP_18_x2['nucleotide_or_aa'] == 'nucleotide']

controls_aa_OP_18 = controls_OP_18_x2[controls_OP_18_x2['nucleotide_or_aa'] == 'aa']

In [28]:
controls_nuc_drop_OP_18 = (controls_nuc_OP_18
    .sort_values(by="VariantNotes_OP", na_position="last") 
    .drop_duplicates(subset=['Gene', 'hg38_start', 'ref_allele', 'alt_allele'], keep="first")
)

In [29]:
controls_nuc_drop_OP_18 = controls_nuc_drop_OP_18[controls_nuc_drop_OP_18['clinvar_star_18_25'].isin(one_plus_stars)]

In [30]:
assay_priority_list = ['BRCA1_Findlay_2018', 'BRCA2_Hu_2024', 'VHL_Buckley_2024',
       'JAG1_Gilbert_2024', 'BARD1_unpublished', 'PALB2_unpublished',
       'SFPQ_unpublished', 'RAD51D_unpublished', 'CTCF_unpublished',
       'BAP1_Waters_2024', 'DDX3X_Radford_2023_cLFC_day15',
       'RHO_Wan_2019', 'RAD51C_Olvera-León_2024_z_score_D4_D14',
       'FKRP_Ma_2024', 'LARGE1_Ma_2024',
       'CARD11_Meitlis_2020_Ibrutinib_no_introns',
       'CARD11_Meitlis_2020_DMSO_no_introns',
       'ASPA_Grønbæk-Thygesen_2024_abundance',
       'ASPA_Grønbæk-Thygesen_2024_toxicity',
       'BRCA1_Adamovich_2022_Cisplatin', 'BRCA1_Adamovich_2022_HDR',
       'CHEK2_Gebbia_2024', 'CRX_Shepherdson_2024', 'F9_Popp_2025_model',
       'G6PD_unpublished', 'GCK_Gersing_2023_complementation',
       'GCK_Gersing_2024_abundance', 'KCNE1_Muhammad_2024_absence_of_WT',
       'KCNE1_Muhammad_2024_potassium_flux',
       'KCNE1_Muhammad_2024_presence_of_WT', 
       'KCNH2_Jiang_2022',
       'KCNH2_Kozek_Glazer_2020', 'KCNH2_O_Neill_2024_surface_expression',
       'MSH2_Jia_2021', 'NDUFAF6_Sung_2024', 'OTC_Lo_2023','PTEN_Matreyek_2018',
       'PTEN_Mighell_2018', 'SCN5A_Glazer_2020',
       'SCN5A_Ma_2024_current_density', 'SGCB_Li_2023',
       'TP53_Fayer_2021_meta', 'TP53_Fortuno_2021_Kato_meta',
       'TSC2_rapgap_unpublished', 'TSC2_tuberin_unpublished',
       'KCNQ4_Zheng_2022_current_homozygous',
       'KCNQ4_Zheng_2022_v12_homozygous']


assay_priority_map = {name: i for i, name in enumerate(assay_priority_list)}

controls_aa_OP_18["assay_priority"] = controls_aa_OP_18["Dataset"].map(assay_priority_map)

controls_aa_OP_18["assay_priority"] = controls_aa_OP_18["assay_priority"].fillna(9999)

/tmp/7612373.1.fowler-login.q/ipykernel_2183752/1537160283.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  controls_aa_OP_18["assay_priority"] = controls_aa_OP_18["Dataset"].map(assay_priority_map)
/tmp/7612373.1.fowler-login.q/ipykernel_2183752/1537160283.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  controls_aa_OP_18["assay_priority"] = controls_aa_OP_18["assay_priority"].fillna(9999)


In [31]:
group_cols_aa_cln = ["Gene", "aa_pos", "aa_ref", "aa_alt","Ref_seq_transcript_ID_stripped"]

one_plus_stars = {
    'criteria provided, single submitter',
    'criteria provided, multiple submitters, no conflicts',
    'reviewed by expert panel',
    'criteria provided, conflicting classifications'
}

zero_stars = {
    'no classification for the single variant',
    'no classification provided','no assertion criteria provided'
}


def summarize_clnstar(series):
    sigs = set(series.dropna())

    # All missing
    if len(sigs) == 0:
        return "Unseen"

    one_star = any(val in one_plus_stars for val in sigs)
    zero_star = any(val in zero_stars for val in sigs)

    if one_star and zero_star:
        return "has_clinvar_star_conflict"

    if one_star:
        return "one_plus_star"

    if zero_star:
        return "zero_star"

    raise ValueError(
        f"Unexpected ClinVar review_status values encountered: {sigs}"
    )

controls_aa_OP_18["clinvar_star_18_25_group"] = (
    controls_aa_OP_18
    .groupby(group_cols_aa_cln)["clinvar_star_18_25"]
    .transform(summarize_clnstar)
)

/tmp/7612373.1.fowler-login.q/ipykernel_2183752/2297292388.py:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  controls_aa_OP_18["clinvar_star_18_25_group"] = (


In [32]:
controls_aa_OP_18 = controls_aa_OP_18[controls_aa_OP_18['clinvar_star_18_25_group'].isin(['one_plus_star'])]

In [33]:
controls_aa_drop_OP_REVEL_Pejaver_18 = (
    controls_aa_OP_18[
        (controls_aa_OP_18['VariantNotes_OP'] == 'max_fxn_pts')
        & (controls_aa_OP_18['GenomeWide_REVEL_max'] == 'max_pred_pts')
    ]
    .sort_values("assay_priority")
    .drop_duplicates(subset=["Gene", "aa_pos", "aa_ref", "aa_alt","Ref_seq_transcript_ID_stripped"], keep="first")
)


controls_aa_drop_OP_AM_Pejaver_18 = (
    controls_aa_OP_18[
        (controls_aa_OP_18['VariantNotes_OP'] == 'max_fxn_pts')
        & (controls_aa_OP_18['GenomeWide_AM_max'] == 'max_pred_pts')
    ]
    .sort_values("assay_priority")
    .drop_duplicates(subset=["Gene", "aa_pos", "aa_ref", "aa_alt","Ref_seq_transcript_ID_stripped"], keep="first")
)


controls_aa_drop_OP_MP2_Pejaver_18 = (
    controls_aa_OP_18[
        (controls_aa_OP_18['VariantNotes_OP'] == 'max_fxn_pts')
        & (controls_aa_OP_18['GenomeWide_MP2_max'] == 'max_pred_pts')
    ]
    .sort_values("assay_priority")
    .drop_duplicates(subset=["Gene", "aa_pos", "aa_ref", "aa_alt","Ref_seq_transcript_ID_stripped"], keep="first")
)

In [34]:
controls_no_dup_REVEL_OP_18_Pejaver = pd.concat([controls_nuc_drop_OP_18,controls_aa_drop_OP_REVEL_Pejaver_18])
controls_no_dup_AM_OP_18_Pejaver = pd.concat([controls_nuc_drop_OP_18,controls_aa_drop_OP_AM_Pejaver_18])
controls_no_dup_MP2_OP_18_Pejaver = pd.concat([controls_nuc_drop_OP_18,controls_aa_drop_OP_MP2_Pejaver_18])

In [35]:
controls_REVEL_18_OP_Pejaver = controls_no_dup_REVEL_OP_18_Pejaver[controls_no_dup_REVEL_OP_18_Pejaver['revel_train_amino'] != 'Yes']
controls_MP2_18_OP_Pejaver = controls_no_dup_MP2_OP_18_Pejaver[controls_no_dup_MP2_OP_18_Pejaver['mp2_train_amino'] != 'Yes']

In [36]:
def catch_mis_2(df, group_cols, points_col='Fxn_points'):
    """
    Handle duplicates by keeping the row with the highest functional points.
    """
    group_cols = ['Gene', 'Chrom', 'hg38_start', 'ref_allele', 'alt_allele']
    df_sorted = df.sort_values(by=points_col, ascending=False, na_position='last')
    cleaned = df_sorted.drop_duplicates(subset=group_cols, keep='first')
    return cleaned

In [37]:
#REVEL

group_cols = ['Gene', 'Chrom', 'hg38_start', 'ref_allele', 'alt_allele']


controls_REVEL_18_OP_Pejaver_cleaned  = catch_mis_2(
    controls_REVEL_18_OP_Pejaver,
    group_cols, points_col='Fxn_points'
)

controls_MP2_18_OP_Pejaver_cleaned  = catch_mis_2(
    controls_MP2_18_OP_Pejaver,
    group_cols, points_col='Fxn_points'
)

controls_AM_18_OP_Pejaver_cleaned  = catch_mis_2(
    controls_no_dup_AM_OP_18_Pejaver,
    group_cols, points_col='Fxn_points'
)

In [38]:
#VUS, check all on the nucleotide level 
VUS_18_OP = sankey_OP_18[sankey_OP_18['clinvar_18_25'].isin(['Uncertain significance'])]

In [39]:
VUS_no_dup_18_OP = (
    VUS_18_OP
    .sort_values(by="VariantNotes_OP", na_position="last") 
    .drop_duplicates(subset=['Gene', 'hg38_start', 'ref_allele', 'alt_allele'], keep="first")
)

In [40]:
VUS_no_dup_REVEL_18_OP = VUS_no_dup_18_OP[VUS_no_dup_18_OP['revel_train_amino'] != 'Yes']

In [41]:
VUS_no_dup_mut_18_OP = VUS_no_dup_18_OP[VUS_no_dup_18_OP['mp2_train_amino'] != 'Yes']

In [42]:
VUS_no_dup_AM_18_OP = VUS_no_dup_18_OP

In [43]:
# Unseen nucleotide variants, sankey_f is already filtered for splice variants not measured, conflicting functional data, and Flags removed, need to remove training variants where appropriate

Unseen = sankey_OP_18[(sankey_OP_18['clinvar_sig_2025'].isna()) & (sankey_OP_18['gnomad_MAF'].isna())]

In [44]:
unseen_no_dup = (
    Unseen
    .sort_values(by="VariantNotes_OP", na_position="last") 
    .drop_duplicates(subset=['Gene', 'hg38_start', 'ref_allele', 'alt_allele'], keep="first")
)

In [45]:
#filter for SNVs
unseen_no_dup = unseen_no_dup[
    (unseen_no_dup['ref_allele'].str.len() == 1) &
    (unseen_no_dup['alt_allele'].str.len() == 1)
]

In [46]:
unseen_no_dup_REVEL = unseen_no_dup[unseen_no_dup['revel_train_amino'] != 'Yes']

unseen_no_dup_mut = unseen_no_dup[unseen_no_dup['mp2_train_amino'] != 'Yes']

unseen_no_dup_AM = unseen_no_dup

In [47]:
gnomad = sankey_OP_18[sankey_OP_18['gnomad_MAF'].notna()]

In [48]:
gnomad_no_dup = (
    gnomad
    .sort_values(by="VariantNotes_OP", na_position="last") 
    .drop_duplicates(subset=['Gene', 'hg38_start', 'ref_allele', 'alt_allele'], keep="first")
)

In [49]:
gnomad_no_dup_REVEL = gnomad_no_dup[gnomad_no_dup['revel_train_amino'] != 'Yes']

gnomad_no_dup_mut = gnomad_no_dup[gnomad_no_dup['mp2_train_amino'] != 'Yes']

gnomad_no_dup_AM = gnomad_no_dup

In [51]:
clingen = sankey_OP_18[sankey_OP_18['Updated_Classification_ClinGen_repo'].notna() & (sankey_OP_18['Updated_Classification_ClinGen_repo'] != 'VUS')]

In [52]:
clingen_nuc = clingen[clingen['nucleotide_or_aa'] == 'nucleotide']

clingen_aa = clingen[clingen['nucleotide_or_aa'] == 'aa']

In [53]:
clingen_nuc_drop = (clingen_nuc
    .sort_values(by="VariantNotes_OP", na_position="last") 
    .drop_duplicates(subset=['Gene', 'hg38_start', 'ref_allele', 'alt_allele'], keep="first")
)

In [54]:
assay_priority_map = {name: i for i, name in enumerate(assay_priority_list)}

clingen_aa["assay_priority"] = clingen_aa["Dataset"].map(assay_priority_map)

clingen_aa["assay_priority"] = clingen_aa["assay_priority"].fillna(9999)

/tmp/7612373.1.fowler-login.q/ipykernel_2183752/2987713055.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clingen_aa["assay_priority"] = clingen_aa["Dataset"].map(assay_priority_map)
/tmp/7612373.1.fowler-login.q/ipykernel_2183752/2987713055.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clingen_aa["assay_priority"] = clingen_aa["assay_priority"].fillna(9999)


In [55]:
clingen_aa_drop_REVEL_Pejaver = (clingen_aa[
        (clingen_aa['VariantNotes'] == 'max_fxn_pts')
        & (clingen_aa['GenomeWide_REVEL_max'] == 'max_pred_pts')
    ]
    .sort_values("assay_priority")
    .drop_duplicates(subset=["Gene", "aa_pos", "aa_ref", "aa_alt","Ref_seq_transcript_ID_stripped"], keep="first")
)

clingen_aa_drop_mut_Pejaver = (clingen_aa[
        (clingen_aa['VariantNotes'] == 'max_fxn_pts')
        & (clingen_aa['GenomeWide_MP2_max'] == 'max_pred_pts')
    ]
    .sort_values("assay_priority")
    .drop_duplicates(subset=["Gene", "aa_pos", "aa_ref", "aa_alt","Ref_seq_transcript_ID_stripped"], keep="first")
)

clingen_aa_drop_AM_Pejaver = (clingen_aa[
        (clingen_aa['VariantNotes'] == 'max_fxn_pts')
        & (clingen_aa['GenomeWide_AM_max'] == 'max_pred_pts')
    ]
    .sort_values("assay_priority")
    .drop_duplicates(subset=["Gene", "aa_pos", "aa_ref", "aa_alt","Ref_seq_transcript_ID_stripped"], keep="first")
)


In [56]:
clingen_no_dup_REVEL_Pejaver = pd.concat([clingen_nuc_drop,clingen_aa_drop_REVEL_Pejaver])
clingen_no_dup_mut_Pejaver = pd.concat([clingen_nuc_drop,clingen_aa_drop_mut_Pejaver])
clingen_no_dup_AM_Pejaver = pd.concat([clingen_nuc_drop,clingen_aa_drop_AM_Pejaver])

In [57]:
ClinGen_repo_REVEL_Pejaver = clingen_no_dup_REVEL_Pejaver[clingen_no_dup_REVEL_Pejaver['revel_train_amino'] != 'Yes']
ClinGen_repo_mut_Pejaver = clingen_no_dup_mut_Pejaver[clingen_no_dup_mut_Pejaver['mp2_train_amino'] != 'Yes']
ClinGen_repo_AM_Pejaver = clingen_no_dup_AM_Pejaver

In [58]:
#REVEL
ClinGen_repo_REVEL_Pejaver_cleaned  = catch_mis_2(
    ClinGen_repo_REVEL_Pejaver,
    group_cols, points_col='Fxn_points'
)

ClinGen_repo_mut_Pejaver_cleaned  = catch_mis_2(
    ClinGen_repo_mut_Pejaver,
    group_cols, points_col='Fxn_points'
)

ClinGen_repo_AM_Pejaver_cleaned  = catch_mis_2(
    ClinGen_repo_AM_Pejaver,
    group_cols, points_col='Fxn_points'
)

In [68]:
dfs = {
    "controls_REVEL_OP": controls_REVEL_18_OP_Pejaver_cleaned,
    "controls_MP2_OP": controls_MP2_18_OP_Pejaver_cleaned,
    "controls_AM_OP": controls_AM_18_OP_Pejaver_cleaned,
    
    "VUS_REVEL_OP" : VUS_no_dup_REVEL_18_OP,
    "VUS_MP2_OP" : VUS_no_dup_mut_18_OP,
    "VUS_AM_OP" : VUS_no_dup_AM_18_OP,

    "ClinGen_Repo_REVEL_OP": ClinGen_repo_REVEL_Pejaver_cleaned,
    "ClinGen_repo_MP2_OP": ClinGen_repo_mut_Pejaver_cleaned,
    "ClinGen_repo_AM_OP": ClinGen_repo_AM_Pejaver_cleaned,
    
    "gnomAD_REVEL_OP": gnomad_no_dup_REVEL,
    "gnomAD_AM_OP": gnomad_no_dup_AM,
    "gnomAD_MP2_OP": gnomad_no_dup_mut,
    
    "Unobserved_REVEL_OP" : unseen_no_dup_REVEL,
    "Unobserved_AM_OP" : unseen_no_dup_AM,
    "Unobserved_mut_OP" : unseen_no_dup_mut
    
    
}


In [70]:
COLUMNS_TO_DROP = ['REVEL_GenomeWide_Code', 'MP2_GenomeWide_Code', 'AM_GenomeWide_Code', 'REVEL_GeneSpecific_Code', 
                   'AM_GeneSpecific_Code', 'MP2_GeneSpecific_Code','Points_REVEL_GeneSpecific', 
                   'Points_AM_GeneSpecific', 'Points_MP2_GeneSpecific', 'Points_REVEL_GeneSpecific_GenomeWide', 
                   'Points_AM_GeneSpecific_GenomeWide', 'Points_MP2_GeneSpecific_GenomeWide', 'Total_Points_GenomeWide_REVEL', 
                   'Total_Points_GenomeWide_AM', 'Total_Points_GenomeWide_MP2', 'Total_Points_GeneSpecific_REVEL',
                   'Total_Points_GeneSpecific_AM', 'Total_Points_GeneSpecific_MP2', 'Class_GenomeWide_REVEL', 'Class_GenomeWide_AM', 
                   'Class_GenomeWide_MP2', 'Class_GeneSpecific_REVEL', 'Class_GeneSpecific_AM', 'Class_GeneSpecific_MP2', 
                   'Conflicting_REVEL_GenomeWide', 'Conflicting_AM_GenomeWide', 'Conflicting_MP2_GenomeWide', 
                   'Conflicting_REVEL_GeneSpecific', 'Conflicting_AM_GeneSpecific', 'Conflicting_MP2_GeneSpecific','splice_variant', 
                   'VariantNotes', 'GenomeWide_REVEL_max', 'GeneSpecific_REVEL_max', 'GenomeWide_AM_max', 'GeneSpecific_AM_max', 
                   'GenomeWide_MP2_max', 'GeneSpecific_MP2_max','clnsig_group_25','revel_train_amino', 'mp2_train_amino',
                   'splice_var_amino', 'clinvar_conflict_flag_18_25', 'VariantNotes_OP', 'Ref_seq_transcript_ID_stripped', 
                   'clinvar_star_18_25', 'assay_priority', 'clinvar_star_18_25_group'
    
]

In [71]:
for name, df in dfs.items():
    dfs[name] = df.drop(columns=COLUMNS_TO_DROP, errors="ignore")

In [72]:
# Define new column names
rename_dict = {
    'Total_Points_OP_GenomeWide_REVEL': 'Total_Points_OP_REVEL',
    'Total_Points_OP_GenomeWide_AM': 'Total_Points_OP_AM',
    'Total_Points_OP_GenomeWide_MP2': 'Total_Points_OP_MP2',
    'ClassOP_GenomeWide_REVEL': 'Class_OP_REVEL', 
    'ClassOP_GenomeWide_AM':'Class_OP_AM',
    'ClassOP_GenomeWide_MP2':'Class_OP_MP2',
    'Conflicting_OP_REVEL_GenomeWide': 'Conflicting_OP_REVEL',
    'Conflicting_OP_AM_GenomeWide': 'Conflicting_OP_AM',
    'Conflicting_OP_MP2_GenomeWide': 'Conflicting_OP_MP2',
    'clinvar_18_25': 'clinvar_sig_18_25'

}

# Define column order
column_order = ['ID', 'Dataset', 'Gene', 'HGNC_id', 'Chrom', 'Strand', 'hg19_pos', 'hg38_start', 'hg38_end', 'ref_allele', 'alt_allele', 
                'auth_transcript_id', 'transcript_pos', 'transcript_ref', 'transcript_alt', 'aa_pos', 'aa_ref', 'aa_alt', 'hgvs_c', 
                'hgvs_p', 'consequence', 'simplified_consequence', 'auth_reported_score', 'auth_reported_rep_score', 
                'auth_reported_func_class', 'splice_measure', 'gnomad_MAF', 'clinvar_sig_2025', 'clinvar_star_2025', 
                'clinvar_date_last_reviewed_2025', 'clinvar_sig_2018', 'clinvar_star_2018', 'clinvar_date_last_reviewed_2018', 
                'nucleotide_or_aa', 'Ensembl Transcript ID', 'RefSeq Transcript ID', 'Interval 1 Name', 'Interval 1 Range',
                'Interval 1 Class', 'Interval 2 Name', 'Interval 2 Range', 'Interval 2 Class', 'Interval 3 Name', 'Interval 3 Range', 
                'Interval 3 Class', 'Interval 4 Name', 'Interval 4 Range', 'Interval 4 Class', 'Interval 5 Name', 'Interval 5 Range', 
                'Interval 5 Class', 'Interval 6 Name', 'Interval 6 Range', 'Interval 6 Class', 'Flag', 'REVEL', 'REVEL_train', 
                'AM_score', 'AM_class', 'MutPred2', 'MP2_train', 'spliceAI_DS_AG', 'spliceAI_DS_AL', 'spliceAI_DS_DG', 'spliceAI_DS_DL',
                'spliceAI_DP_AG', 'spliceAI_DP_AL', 'spliceAI_DP_DG', 'spliceAI_DP_DL', 'ClinVar Variation Id_ClinGen_repo', 
                'Allele Registry Id_ClinGen_repo', 'Disease_ClinGen_repo', 'Mondo Id_ClinGen_repo', 'Mode of Inheritance_ClinGen_repo', 
                'Assertion_ClinGen_repo', 'Applied Evidence Codes (Met)_ClinGen_repo', 'Applied Evidence Codes (Not Met)_ClinGen_repo', 
                'Summary of interpretation_ClinGen_repo', 'PubMed Articles_ClinGen_repo', 'Expert Panel_ClinGen_repo', 
                'Guideline_ClinGen_repo', 'Approval Date_ClinGen_repo', 'Published Date_ClinGen_repo', 'Retracted_ClinGen_repo', 
                'Evidence Repo Link_ClinGen_repo', 'Uuid_ClinGen_repo', 'Updated_Classification_ClinGen_repo', 
                'Updated_Evidence Codes_ClinGen_repo','clinvar_sig_18_25','clnsig_group_18_25','StandardizedClass','ExC_points_2025', 
                'ExC_points_2018', 'OddsNormal', 'OddsAbnormal', 'OP_points', 'Fxn_points', 'Points_REVEL_GenomeWide', 
                'Points_AM_GenomeWide', 'Points_MP2_GenomeWide', 'Total_Points_OP_REVEL',
                'Total_Points_OP_AM', 'Total_Points_OP_MP2', 'Class_OP_REVEL', 
                'Class_OP_AM', 'Class_OP_MP2', 'Conflicting_OP_REVEL', 'Conflicting_OP_AM',
                'Conflicting_OP_MP2']

# Apply to all dataframes in dictionary
for key in dfs:
    dfs[key] = dfs[key].rename(columns=rename_dict)[column_order]

In [74]:
out_folder = OUT_DIR / "OddsPath_calibrations"
out_folder.mkdir(exist_ok=True)

for name, df in dfs.items():
    df.to_csv(out_folder / f"{name}.csv", index=False)